# IPCA noise robustness: historical experimental configuration

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Defining the utils**

**COMMENT: restart after having run this cell, so it is in memory**

**Description**

1. Change the results path for the number hidden layer in each model, iteratively : results_CA_0, results_CA_1, results_CA_2, results_CA_0
2. Run the code that defines the desired model with the number of hidden layers
3. Run the codes that preprocess the data for the models.
4. Run the code for the cross-validation, which makes prediction on the validation set. The performance evaluation metrcs are saved in the results path.Remember to change the results path for the cross-validation.
5. Run the the code that predicts the desired model with the desired number of hidden layers. remember to change the results path for the out-of-sample  predictions on the test set. This code makes predictions for each latent latent factor across a time-series split, and the predicted results are saved.

**sliding window with validation**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            train_end_idx = test_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits


In [ ]:
!pip install tables

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time
import random
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from utils import format_time, MultipleTimeSeriesCV

from numpy.linalg import inv
from google.colab import drive





**defining my results path**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/Robustness_check_total_IPCA')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)


In [ ]:

###########################
## MOUNT TO GOOGLE DRIVE ##
###########################

# Mount Google Drive
drive.mount('/content/drive')

################################
## LOAD DATA FOR BETA NETWORK ##
################################


file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'

# Make sure merged_df1 is a DataFrame and not a function or any other type.
merged_df_padded = pd.read_parquet(file_path)

# Assuming merged_df_padded is already defined
# Ensure the date level of the MultiIndex is of datetime type
merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)

# Filter the DataFrame to include only rows from 1979 and onwards
merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

print(merged_df_padded)
print(merged_df_padded.index)
print(merged_df_padded.shape)
print(type(merged_df_padded))

#
#################################
## LOAD DATA FOR FACTOR NETWORK##
#################################

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/x_t_preprocessed.parquet'

# Load the Parquet file into a DataFrame
portfolio_rets_df = pd.read_parquet(file_path)



# Assuming portfolio_rets_df is already defined and loaded
# Ensure the date index is of datetime type
portfolio_rets_df.index = pd.to_datetime(portfolio_rets_df.index)

# Filter the DataFrame to include only rows from 1979 and onwards
portfolio_rets_df = portfolio_rets_df.loc[portfolio_rets_df.index >= '1960-01-01']

print(portfolio_rets_df)
print(portfolio_rets_df.index)
print(portfolio_rets_df.shape)
print(type(portfolio_rets_df))




In [ ]:
# import pandas as pd
# import numpy as np

# # Mount Google Drive
# drive.mount('/content/drive')

# # Load data for beta network
# beta_file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
# merged_df_padded = pd.read_parquet(beta_file_path)
# merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
# merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1975-01-01']

# # Remove rows with any -2 values
# merged_df_cleaned = merged_df_padded[~(merged_df_padded == -2).any(axis=1)]




# print(merged_df_cleaned)
# print(merged_df_cleaned.index)
# print(merged_df_cleaned.shape)
# print(type(merged_df_cleaned))


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade", "equal_weighted_portfolio"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Assuming merged_df_padded is already defined and loaded

# Replace -2 with NaN to correctly handle missing values
merged_df_padded.replace(-2, np.nan, inplace=True)

# Parameters for noise generation
unique_permnos = merged_df_padded.index.get_level_values('permno').unique()
unique_dates = merged_df_padded.index.get_level_values('date').unique()
N = len(unique_permnos)  # Number of unique permnos
T = len(unique_dates)  # Number of time periods
Pc = 5  # Number of noise characteristics

# Calculate the standard deviation of the specified columns ignoring NaNs
characteristics_std = merged_df_padded[characteristics].std(skipna=True).mean()

# Simulate noise characteristics
rho = np.random.uniform(0.9, 1, Pc)
c = np.zeros((N, Pc, T))

# Generate the noise characteristics
for t in tqdm(range(1, T), desc="Generating noise characteristics"):
    c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

# Normalize noise characteristics
c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

# Scale noise characteristics
c_scaled = c_normalized * characteristics_std

# Check the shape of c_scaled
print(c_scaled.shape)  # (N, Pc, T)

# Reshape scaled characteristics to fit the DataFrame structure
c_scaled = c_scaled.transpose(2, 0, 1)  # Transpose to (T, N, Pc)
c_scaled = c_scaled.reshape(T * N, Pc)  # Flatten to (T*N, Pc)

# Create a new MultiIndex to match the merged_df_padded index
dates = np.repeat(unique_dates, N)
permnos = np.tile(unique_permnos, T)

# Verify the lengths of dates and permnos arrays
print(f'Number of dates: {len(dates)}')  # Should be T * N
print(f'Number of permnos: {len(permnos)}')  # Should be T * N

# Create the new MultiIndex
new_index = pd.MultiIndex.from_arrays([dates, permnos], names=['date', 'permno'])

# Create the DataFrame
noise_features_df = pd.DataFrame(c_scaled, index=new_index, columns=[f'noise_{i}' for i in range(Pc)])

# Ensure the new index matches the original DataFrame index
noise_features_df = noise_features_df.loc[merged_df_padded.index]

# Add noise features to the original characteristics
beta_with_noise = pd.concat([merged_df_padded, noise_features_df], axis=1)

# Replace NaNs back with -2 to maintain the same structure as the original data
beta_with_noise.fillna(-2, inplace=True)

# Swap the levels of the MultiIndex
beta_with_noise = beta_with_noise.swaplevel('date', 'permno')

# Optionally, sort the index to ensure correct ordering
beta_with_noise = beta_with_noise.sort_index()

# Remove rows where any column has a value of -2
beta_with_noise = beta_with_noise[~(beta_with_noise == -2).any(axis=1)]

# Print to check the DataFrame
print(beta_with_noise.head())
print(beta_with_noise.shape)
print(type(beta_with_noise))





In [ ]:
# import numpy as np
# import pandas as pd

# # Assuming portfolio_rets_df is already defined and loaded

# # Define the list of characteristic codes (same as before)
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Parameters for noise generation
# unique_dates_portfolio = portfolio_rets_df.index.unique()
# T_portfolio = len(unique_dates_portfolio)  # Number of time periods for portfolio returns
# Pc_portfolio = 5  # Number of noise portfolios

# # Calculate the standard deviation of the specified columns
# characteristics_std_portfolio = portfolio_rets_df[characteristics].std().mean()

# # Simulate noise portfolios
# rho_portfolio = np.random.uniform(0.9, 1, Pc_portfolio)
# c_portfolio = np.zeros((T_portfolio, Pc_portfolio))

# # Generate the noise portfolios
# for t in range(1, T_portfolio):
#     c_portfolio[t, :] = rho_portfolio * c_portfolio[t - 1, :] + np.random.normal(0, 1, Pc_portfolio)

# # Normalize noise portfolios
# c_normalized_portfolio = 2 / (T_portfolio + 1) * (c_portfolio.argsort(axis=0).argsort(axis=0) - 1) - 1

# # Scale noise portfolios
# c_scaled_portfolio = c_normalized_portfolio * characteristics_std_portfolio

# # Check the shape of c_scaled_portfolio
# print(c_scaled_portfolio.shape)  # (T_portfolio, Pc_portfolio)

# # Create a DataFrame for noise portfolios with the same index as portfolio_rets_df
# noise_portfolios_df = pd.DataFrame(c_scaled_portfolio, index=portfolio_rets_df.index, columns=[f'noise_{i}' for i in range(Pc_portfolio)])

# # Add noise portfolios to the original portfolio returns
# portfolios_with_noise = pd.concat([portfolio_rets_df, noise_portfolios_df], axis=1)

# # Print to check the DataFrame
# print(portfolios_with_noise.head())
# print(portfolios_with_noise.shape)
# print(type(portfolios_with_noise))



In [ ]:

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Select the required columns including the noise characteristics
# columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)] + ["ret_excess"]
# characteristics_data = beta_with_noise[columns_to_select]



# # # Select the required columns including the noise characteristics
# # characteristics = list(merged_df_cleaned.columns)
# # columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)] + ["ret_excess"]
# # characteristics_data = beta_with_noise[columns_to_select]

# # Function to calculate x_t for a given month using Ridge Regression
# def calculate_xt(df, date, alpha=0):
#     # Filter DataFrame for the given date
#     df_month = df.loc[date]

#     # Separate returns and characteristics
#     r_t = df_month['ret_excess'].values
#     characteristic_columns = df_month.drop(columns=['ret_excess']).columns
#     Z_t_minus_1 = df_month.drop(columns=['ret_excess']).values

#     # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
#     Z_t_minus_1_T = Z_t_minus_1.T
#     Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
#     Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

#     Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

#     # Calculate x_t
#     x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

#     return x_t, characteristic_columns

# # Calculate x_t for all unique dates in the dataset
# def calculate_xt_all_dates(df, alpha=1e-6):
#     xt_results = {}
#     characteristic_columns = None

#     unique_dates = df.index.get_level_values('date').unique()
#     for date in unique_dates:
#         x_t, characteristic_columns = calculate_xt(df, date, alpha)
#         xt_results[date] = x_t

#     # Convert dictionary to DataFrame
#     xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

#     # Assign original characteristic names as column names
#     if characteristic_columns is not None:
#         xt_df.columns = characteristic_columns

#     # Preserve the index name
#     xt_df.index.name = 'date'

#     return xt_df

# # Example usage
# xt_all_dates = calculate_xt_all_dates(characteristics_data)


**MODEL PARAMETER SPECIFICATIONS**

In [ ]:
# # efine the save path
# load_path = '/content/drive/My Drive/Tidy Finance ML dataset'

# # Save the updated data to parquet files
# beta_with_noise.to_parquet(f'{load_path}/final_df.parquet', index=True)
# portfolios_with_noise.to_parquet(f'{load_path}/xt_all_dates.parquet', index=True)

# print("DataFrames have been saved successfully as parquet files.")

In [ ]:
# 'import pandas as pd

# # Define the load path
# load_path = '/content/drive/My Drive/Tidy Finance ML dataset'

# # Load the parquet files
# beta_with_noise = pd.read_parquet(f'{load_path}/final_df.parquet')
# portfolios_with_noise = pd.read_parquet(f'{load_path}/xt_all_dates.parquet')

# # Remove rows where any column has a value of -2
# beta_with_noise = beta_with_noise[~(beta_with_noise == -2).any(axis=1)]

# # Print the loaded dataframes to verify
# print("Loaded final_df:")
# print(beta_with_noise.info())
# print(beta_with_noise.head())

# print("Loaded xt_all_dates:")
# print(portfolios_with_noise.info())
# print(portfolios_with_noise.head())
# portfolios_with_noise = portfolios_with_noise.drop(columns=['characteristic_equal_weighted_portfolio'])'

In [ ]:
# import pandas as pd

# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# # Define the path to load the files
# load_path = '/content/drive/My Drive/Tidy Finance ML dataset/'

# # Load final_df from the HDF5 file
# beta_with_noise = pd.read_hdf(f'{load_path}/final_df.h5', key='final_df')

# # Load xt_all_dates from the HDF5 file
# portfolios_with_noise = pd.read_hdf(f'{load_path}/xt_all_dates.h5', key='xt_all_dates')

# # Verify the loaded data
# print(beta_with_noise.info())
# print(beta_with_noise.head())
# portfolios_with_noise = portfolios_with_noise.drop(columns=['characteristic_equal_weighted_portfolio'])

# print(portfolios_with_noise.info())
# print(portfolios_with_noise.head())


In [ ]:
Pc=5
# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Select the required columns including the noise characteristics
columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)]

narratives = columns_to_select
print(narratives)


n_factors = 6
n_narratives = len(columns_to_select)
n_tickers = len(beta_with_noise.index.unique('permno'))

print(n_narratives)
print(n_tickers)



In [ ]:

# Define the columns to select for final_df
columns_to_select_beta = narratives + ["ret_excess", "fwd_ret_excess"]

# Convert specified columns to float16 directly in final_df
for column in columns_to_select_beta:
    beta_with_noise[column] = beta_with_noise[column].astype('float16')

print("Updated final_df:")
print(beta_with_noise.info())
print(beta_with_noise.head())

# Define the columns to select for xt_all_dates
columns_to_select_portfolios = narratives



**Converting data to float16 for efficiency and saving files**

In [ ]:
# import pandas as pd


# # Define the columns to select
# columns_to_select = narratives + ["ret_excess", "fwd_ret_excess"]

# # Convert specified columns to float16 directly in final_df
# for column in columns_to_select:
#     beta_with_noise[column] = beta_with_noise[column].astype('float16')

# print(beta_with_noise)


# # Define the columns to select
# columns_to_select = narratives

# # Convert specified columns to float16 directly in xt_all_dates
# for column in columns_to_select:
#     portfolios_with_noise[column] = portfolios_with_noise[column].astype('float16')

# print(portfolios_with_noise)



**Saving the figures**

In [ ]:
!pip install ipca



```
# This is formatted as code
```

#**training loop**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.utils import shuffle
from ipca import InstrumentedPCA
from tqdm import tqdm

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]
    print(train_set.index)
    print(test_set.index)

    y_train = train_set['fwd_ret_excess'].to_numpy()
    x_train = train_set.drop(columns=['fwd_ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['fwd_ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    # Return the processed data
    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

def compute_permutation_variable_importance_ipca(model, X, y):
    # Get the baseline performance
    preds = model.predict(X=X, mean_factor=True)
    baseline_r2 = r2_score(y, preds)

    importances = []

    # Iterate over each feature
    for i in range(X.shape[1]):
        X_copy = X.copy()

        # Shuffle the i-th feature, retaining the index
        X_copy.iloc[:, i] = np.random.permutation(X_copy.iloc[:, i].values)

        preds_permuted = model.predict(X=X_copy, mean_factor=True)
        permuted_r2 = r2_score(y, preds_permuted)

        importance = baseline_r2 - permuted_r2
        importances.append(importance)

    return importances


# def calculate_xt(beta_with_noise, result, columns_to_select):
#     """
#     Calculate x_t for each date using the given characteristics and returns data.

#     Parameters:
#     - beta_with_noise: DataFrame containing stock characteristics with MultiIndex (date, permno).
#     - result: DataFrame containing predicted and true returns with MultiIndex (date, permno).
#     - columns_to_select: List of column names representing stock characteristics.

#     Returns:
#     - xt_df_true: DataFrame containing x_t for each date using true returns.
#     - xt_df_pred: DataFrame containing x_t for each date using predicted returns.
#     """
#     xt_true_list = []
#     xt_pred_list = []

#     # Ensure index is sorted by date for processing
#     beta_with_noise = beta_with_noise.sort_index(level=0)
#     result = result.sort_index(level=0)

#     # Filter beta_with_noise to only include dates present in result
#     common_dates = result.index.get_level_values(0).unique()
#     beta_with_noise = beta_with_noise.loc[common_dates]

#     for date in common_dates:
#         try:
#             print(f"Processing date: {date}")

#             # Extract Z_{t-1} for the given date
#             Z_t_1 = beta_with_noise.loc[date, columns_to_select].replace(-2, np.nan).dropna()

#             # Extract true and predicted returns for the given date
#             r_t_true = result.loc[date, 'y_true'].dropna()
#             r_t_pred = result.loc[date, 'y_pred'].dropna()

#             # Ensure Z_t_1 and r_t are not empty and have matching dimensions
#             if not Z_t_1.empty and not r_t_true.empty and len(Z_t_1) == len(r_t_true):
#                 # Convert to numpy arrays with float32
#                 Z_t_1 = Z_t_1.astype(np.float32).values
#                 r_t_true = r_t_true.astype(np.float32).values
#                 r_t_pred = r_t_pred.astype(np.float32).values

#                 # Compute (Z'_{t-1} Z_{t-1})^{-1}
#                 Z_t_1_T = Z_t_1.T
#                 Z_t_1_Z_t_1_inv = np.linalg.inv(Z_t_1_T @ Z_t_1)

#                 # Compute x_t using true returns
#                 x_t_true = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_true
#                 xt_true_list.append(pd.DataFrame(x_t_true.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_true.shape[0])]))

#                 # Compute x_t using predicted returns
#                 x_t_pred = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_pred
#                 xt_pred_list.append(pd.DataFrame(x_t_pred.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_pred.shape[0])]))

#             else:
#                 print(f"Skipping date {date} due to mismatched dimensions or empty data.")

#         except Exception as e:
#             print(f"Error processing date {date}: {e}")
#             continue

#     # Concatenate results into DataFrames
#     xt_df_true = pd.concat(xt_true_list)
#     xt_df_pred = pd.concat(xt_pred_list)

#     return xt_df_true, xt_df_pred


def calculate_xt(beta_with_noise, result, columns_to_select):
    """
    Calculate x_t for each date using the given characteristics and returns data.

    Parameters:
    - beta_with_noise: DataFrame containing stock characteristics with MultiIndex (permno, date).
    - result: DataFrame containing predicted and true returns with MultiIndex (permno, date).
    - columns_to_select: List of column names representing stock characteristics.

    Returns:
    - xt_df_true: DataFrame containing x_t for each date using true returns.
    - xt_df_pred: DataFrame containing x_t for each date using predicted returns.
    """
    # Reindex to 'date', 'permno' for easier processing
    beta_with_noise = beta_with_noise.reorder_levels(['date', 'permno']).sort_index(level=0)
    result = result.reorder_levels(['date', 'permno']).sort_index(level=0)

    xt_true_list = []
    xt_pred_list = []

    # Filter beta_with_noise to only include dates present in result
    common_dates = result.index.get_level_values(0).unique()
    beta_with_noise = beta_with_noise.loc[common_dates]

    for date in common_dates:
        try:
            print(f"Processing date: {date}")

            # Extract Z_{t-1} for the given date
            Z_t_1 = beta_with_noise.loc[date, columns_to_select].replace(-2, np.nan).dropna()

            # Extract true and predicted returns for the given date
            r_t_true = result.loc[date, 'y_true'].dropna()
            r_t_pred = result.loc[date, 'y_pred'].dropna()

            # Ensure Z_t_1 and r_t are not empty and have matching dimensions
            if not Z_t_1.empty and not r_t_true.empty and len(Z_t_1) == len(r_t_true):
                # Convert to numpy arrays with float32
                Z_t_1 = Z_t_1.astype(np.float32).values
                r_t_true = r_t_true.astype(np.float32).values
                r_t_pred = r_t_pred.astype(np.float32).values

                # Compute (Z'_{t-1} Z_{t-1})^{-1}
                Z_t_1_T = Z_t_1.T
                Z_t_1_Z_t_1_inv = np.linalg.inv(Z_t_1_T @ Z_t_1)

                # Compute x_t using true returns
                x_t_true = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_true
                xt_true_list.append(pd.DataFrame(x_t_true.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_true.shape[0])]))

                # Compute x_t using predicted returns
                x_t_pred = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_pred
                xt_pred_list.append(pd.DataFrame(x_t_pred.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_pred.shape[0])]))

            else:
                print(f"Skipping date {date} due to mismatched dimensions or empty data.")

        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue

    # Concatenate results into DataFrames
    xt_df_true = pd.concat(xt_true_list)
    xt_df_pred = pd.concat(xt_pred_list)

    return xt_df_true, xt_df_pred


YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22* YEAR, test_period_length=1* YEAR, lookahead=0 * YEAR)
cv_dataset_splits = list(cv_portfolio.split(beta_with_noise))

n_factors = 5  # Example: number of factors to use in the IPCA model
# results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')


# Initialize lists to store importances and R² scores
fold_importances_list = []
r2_scores = []
r2_scores_list = []
r2_xt_scores_list = []  # List to store R² for xt for each fold

for fold, (train_idx, test_idx) in tqdm(enumerate(cv_dataset_splits)):
    x_train, y_train, x_test, y_test = get_train_test_data(beta_with_noise, train_idx, test_idx)

    # Initialize the IPCA model
    ipca_model = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')

    # Fit the model on the training data
    ipca_model.fit(X=x_train, y=y_train)

    # Make predictions on the test data
    y_pred = ipca_model.predict(X=x_test, mean_factor=True)

    # Calculate R² score
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    print(f'Fold {fold+1} - R² score: {r2}')

    # Save the R² score to a DataFrame
    r2_scores_list.append({'fold': fold+1, 'r2_score': r2})

    # Create a DataFrame with the predictions and true values
    test_result = pd.DataFrame({
        'y_pred': y_pred.flatten(),
        'y_true': y_test.flatten()
    }, index=x_test.index)

    print(test_result)

    # Assuming beta_with_noise and result DataFrames are already defined and available
    xt_df_true_test, xt_df_pred_test = calculate_xt(beta_with_noise, test_result, columns_to_select)

    print("TRUE")
    print(xt_df_true_test)
    print("PRED")
    print(xt_df_pred_test)

    # Calculate R² for each column in xt
    r2_xt_columns = []
    for col in xt_df_true_test.columns:
        r2_xt = r2_score(xt_df_true_test[col], xt_df_pred_test[col])
        r2_xt_columns.append(r2_xt)

    # Calculate the average R² score for xt across all columns
    avg_r2_xt = np.mean(r2_xt_columns)
    print(f'Fold {fold+1} - Average R² score for xt: {avg_r2_xt}')

    # Save the average R² score for xt to the list
    r2_xt_scores_list.append({'fold': fold+1, 'avg_r2_xt': avg_r2_xt})



    # Calculate feature importance using permutation
    importances = compute_permutation_variable_importance_ipca(ipca_model, x_test, y_test)
    fold_importances_list.append(importances)

    print(f'Fold {fold+1} - Computed feature importances.')

# Convert the list of average R² scores for xt to a DataFrame and save it
r2_xt_scores_df = pd.DataFrame(r2_xt_scores_list)
r2_xt_scores_df.to_hdf(str(results_path / 'r2_xt_scores.h5'), key='fold', append=True, format='table')


# Save R² scores to HDF5
r2_scores_df = pd.DataFrame(r2_scores_list)
r2_scores_df.to_hdf(str(results_path / 'r2_scores.h5'), key='fold', append=True, format='table')

# Aggregate importances across all folds
avg_importances = np.mean(fold_importances_list, axis=0)
importances_df = pd.DataFrame(avg_importances, index=x_train.columns, columns=['importance'])
importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})

# Save the importance DataFrame
importances_df.to_hdf(str(results_path / 'variable_importance_ipca.h5'), key='all_folds', append=True, format='table')

# Sort and select the top 20 variable importances
importances_df = importances_df.sort_values('importance_normalized', ascending=False).head(20)

# Reverse the DataFrame so that the highest importance is at the top in the plot
importances_df = importances_df.iloc[::-1]

# Plot the top 20 variable importances
importances_df.plot(kind='barh', x='variable', y='importance_normalized', legend=False)
plt.xlabel('Normalized Importance')
plt.ylabel('Variables')
plt.title(f'Top 20 Normalized Characteristics Importances - IPCA Model')
plt.savefig(str(results_path / f'top_20_importances_ipca.png'))
plt.show()

# Calculate and print the average R² score across all folds
avg_r2 = np.mean(r2_scores)
print(f'Average R² score across all folds: {avg_r2}')

In [ ]:
print(importances_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

# Load the saved importances
importances_df = pd.read_hdf(str(results_path / 'variable_importance_ipca.h5'), key='all_folds')

# Remove the 'Characteristic_' prefix from each variable name, except for 'fwd_ret_excess'
importances_df['variable'] = importances_df['variable'].apply(
    lambda x: x.replace('characteristic_', '') if 'characteristic_' in x and x != 'ret_excess' else x
)

# Remove 'fwd_ret_excess' from the dataframe
importances_df = importances_df[importances_df['variable'] != 'ret_excess']

# Sort and select the top 20 variable importances
importances_df = importances_df.sort_values('importance_normalized', ascending=False).head(20)

# Reverse the DataFrame so that the highest importance is at the top in the plot
importances_df = importances_df.iloc[::-1]

# Plot the top 20 variable importances
importances_df.plot(kind='barh', x='variable', y='importance_normalized', legend=False)
plt.xlabel('Normalized Importance')
plt.ylabel('Variables')


# Save the plot directly in the current directory
plt.savefig('top_20_importances_ipca_modified.png')
plt.show()

# Download the plot
files.download('top_20_importances_ipca_modified.png')


In [ ]:
import pandas as pd

# Load the R² scores for predictions
r2_scores_df = pd.read_hdf(str(results_path / 'r2_scores.h5'), key='fold')

# Load the average R² scores for xt
r2_xt_scores_df = pd.read_hdf(str(results_path / 'r2_xt_scores.h5'), key='fold')

# Display the loaded dataframes to verify
print("R² Scores for Predictions:")
print(r2_scores_df)

print("\nAverage R² Scores for xt:")
print(r2_xt_scores_df)
